In [156]:
import os 


In [157]:
%pwd

'c:\\Users\\LENOVO'

In [158]:
from pathlib import Path
import sys
# Ensure local `src` package is used and clear any cached modules.
repo_root = Path.cwd()
sys.path.insert(0, str(repo_root / 'src'))
for module_name in list(sys.modules):
    if module_name.startswith('cnnClassifier'):
        del sys.modules[module_name]

In [159]:
from dataclasses import dataclass 
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class TrainingConfig:
	root_dir: Path 
	trained_model_path: Path
	updated_base_model_path: Path 
	training_data: Path 
	params_epochs: int
	params_batch_size: int
	params_is_augmentation: bool
	params_image_size: list

	

In [160]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
import tensorflow as tf

In [161]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_training_config(self) -> TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params.get("TrainingArguments", self.params.get("training_arguments", self.params))
        training_params = training.get("params", {}) if isinstance(training, dict) else training.params

        if isinstance(training, dict):
            training_data = Path(training.get("training_data", self.config.data_ingestion.unzip_dir))
            trained_model_path = Path(training.get("trained_model_path", "artifacts/training/model.h5"))
            root_dir = Path(training.get("root_dir", "artifacts/training"))
        else:
            training_data = Path(training.training_data)
            trained_model_path = Path(training.trained_model_path)
            root_dir = Path(training.root_dir)

        create_directories([
            root_dir
        ])

        training_config = TrainingConfig(
            root_dir=root_dir,
            trained_model_path=trained_model_path,
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=training_data,
            params_epochs=training_params.get("EPOCHS", training_params.get("epochs", params.get("epochs", params.get("EPOCHS")))),
            params_batch_size=training_params.get("BATCH_SIZE", training_params.get("batch_size", params.get("batch_size", params.get("BATCH_SIZE")))),
            params_is_augmentation=training_params.get("AUGMENTATION", training_params.get("augmentation", params.get("augmentation", params.get("AUGMENTATION")))),
            params_image_size=training_params.get("IMAGE_SIZE", training_params.get("image_size", params.get("image_size", params.get("IMAGE_SIZE"))))
        )
        return training_config

In [162]:
print('CONFIG_FILE_PATH ->', CONFIG_FILE_PATH, type(CONFIG_FILE_PATH), repr(CONFIG_FILE_PATH))

CONFIG_FILE_PATH -> C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\config\config.yaml <class 'pathlib.WindowsPath'> WindowsPath('C:/Users/LENOVO/Kidney-Disease-Classification-DL-Project-DVC/config/config.yaml')


In [163]:
import inspect
import cnnClassifier.constants as const_mod
print('constants module file:', getattr(const_mod, '__file__', None))

constants module file: C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\src\cnnClassifier\constants\__init__.py


In [164]:
import sys
for i, p in enumerate(sys.path[:20]):
    print(i, p)

0 c:\Users\LENOVO\src
1 c:\Users\LENOVO\src
2 c:\Users\LENOVO\src
3 c:\Users\LENOVO\src
4 c:\Users\LENOVO\src
5 C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\src
6 d:\Software\Anaconda\envs\kidney-disease\python310.zip
7 d:\Software\Anaconda\envs\kidney-disease\DLLs
8 d:\Software\Anaconda\envs\kidney-disease\lib
9 d:\Software\Anaconda\envs\kidney-disease
10 
11 d:\Software\Anaconda\envs\kidney-disease\lib\site-packages
12 C:\Users\LENOVO\Documents\Codex\2026-06-04\krishnaik06-kidney-disease-classification-deep-learning\Kidney-Disease-Classification-Deep-Learning-Project\src
13 d:\Software\Anaconda\envs\kidney-disease\lib\site-packages\win32
14 d:\Software\Anaconda\envs\kidney-disease\lib\site-packages\win32\lib
15 d:\Software\Anaconda\envs\kidney-disease\lib\site-packages\Pythonwin
16 d:\Software\Anaconda\envs\kidney-disease\lib\site-packages\setuptools\_vendor


In [165]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

In [166]:
class Training:
    def __init__(self, config: TrainingConfig):
        self.config = config

    
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )

    def train_valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )
        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs
            )
        else:
            train_datagenerator = valid_datagenerator

        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="training",
            shuffle=True,
            **dataflow_kwargs
        )

    
    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)



    
    def train(self):
        self.steps_per_epoch = self.train_generator.samples // self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples // self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps=self.validation_steps,
            validation_data=self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )

In [167]:
import os
from pathlib import Path

# List all artifacts
artifacts = Path("artifacts")
for p in artifacts.rglob("*"):
    print(p)

artifacts\training


In [170]:
try:
    config = ConfigurationManager()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
    
except Exception as e:
	try:
		os.chdir(ROOT_DIR)
		config = ConfigurationManager()
		training_config = config.get_training_config()
		training = Training(config=training_config)
		training.get_base_model()
		training.train_valid_generator()
		training.train()
	except Exception:
		raise

[2026-06-11 04:20:59,662: INFO: common: yaml file: C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\config\config.yaml loaded successfully]
[2026-06-11 04:20:59,668: INFO: common: yaml file: C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\params.yaml loaded successfully]
[2026-06-11 04:20:59,670: INFO: common: created directory at: artifacts]
[2026-06-11 04:20:59,673: INFO: common: created directory at: artifacts\training]
[2026-06-11 04:20:59,677: INFO: common: yaml file: C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\config\config.yaml loaded successfully]
[2026-06-11 04:20:59,680: INFO: common: yaml file: C:\Users\LENOVO\Kidney-Disease-Classification-DL-Project-DVC\params.yaml loaded successfully]
[2026-06-11 04:20:59,683: INFO: common: created directory at: artifacts]
[2026-06-11 04:20:59,684: INFO: common: created directory at: artifacts\training]
Found 210 images belonging to 2 classes.
Found 844 images belonging to 2 classes.
Epoch 1/5
52/5